# 实验 11–12 · 综合应用实验 4：供给系统问题


实验目标是把供给站的选址问题转化为算法，验证所求的站点数量最少，并推导时间复杂度。

## 题目与输入输出

把 $N$ 个亮区视为平面上的点 $P_i=(X_i,Y_i)$。供给站只能设在 X 轴上。根据输入参数 $R$ 和约束 $|Y_i|\le R$，本笔记按半径 $R$ 的欧氏距离覆盖建模。站点 $(s,0)$ 能覆盖 $P_i$ 的条件是 $(s-X_i)^2+Y_i^2\le R^2$。求覆盖全部亮区的最少站数。

| 顺序 | 标准输入 |
| --- | --- |
| 首行 | 测试组数 `K` |
| 每组首行 | 亮区数 `N` 和供给半径 `R` |
| 随后 `N` 行 | 每行一个亮区的坐标 `Xi Yi`，同行数据用空格分隔 |

每组输出一行，仅含最少站数。手册给出的范围为 $1\le N\le100$、$1\le R\le50$、$-100\le X_i,Y_i\le100$、$|Y_i|\le R$。手册在输入处使用 `K`，在约束处写作 `2 ≤ M ≤ 5`，未另行定义 `M`。读取时以首行实际给出的组数为准。

## 算法思路与伪代码

对亮区 $(X_i,Y_i)$，站点可以左右移动的最大水平距离为 $d_i=\sqrt{R^2-Y_i^2}$。可选的站点横坐标构成闭区间 $[X_i-d_i,X_i+d_i]$。当 $|Y_i|=R$ 时，区间退化为单点 $[X_i,X_i]$。例如，$[-3,3]$ 和 $[3,9]$ 可以共用站点 $(3,0)$。

**贪心选择与正确性**

在尚未覆盖的区间中，取右端点最小的区间 $A$，记其右端点为 $r$。任一最优方案都要在 $A$ 中放置某个站点 $s$。对该站原来覆盖的任一区间 $B=[L_B,R_B]$，有 $L_B\le s\le r\le R_B$，其中最后一步来自 $r$ 是最小右端点。因此把 $s$ 移到 $r$ 仍能覆盖这些区间，站数也不会增加。对剩余未覆盖的区间重复此选择，即可得到最少站数。

**伪代码**

输入为一组亮区坐标和供给半径，输出为覆盖所有亮区的最少站数。手册保证 $N\ge 1$。

```text
对于每个亮区 (Xi, Yi)：
    d ← √(R² - Yi²)
    将闭区间 [Xi - d, Xi + d] 加入区间数组

按右端点从小到大排序
last ← 第一个区间的右端点
count ← 1

对于剩余的每个区间 [L, R]：  // 此处 R 表示当前区间的右端点
    如果 L > last：          // L 等于 last 时可以沿用已有站点
        last ← R
        count ← count + 1

返回 count
```

## C23 实现

先完成求解单组数据的函数，再编写标准输入输出。

In [1]:
#include <math.h>
#include <stdio.h>
#include <stdlib.h>

typedef struct {
    double x;
    double y;
} BrightRegion;

In [2]:
typedef struct {
    double left;
    double right;
} Interval;

static Interval coverage_interval(BrightRegion point, double radius)
{
    double d = sqrt(radius * radius - point.y * point.y);
    return (Interval){
        .left = point.x - d,
        .right = point.x + d
    };
}

In [3]:
static int compare_by_right(const void *first, const void *second)
{
    const Interval *a = first;
    const Interval *b = second;

    if (a->right < b->right) return -1;
    if (a->right > b->right) return 1;
    return 0;
}

In [4]:
int minimum_stations(const BrightRegion regions[], int n, double radius)
{
    Interval intervals[100];  // 手册保证 1 ≤ n ≤ 100
    for (int i = 0; i < n; i++) {
        intervals[i] = coverage_interval(regions[i], radius);
    }
    qsort(intervals, n, sizeof intervals[0], compare_by_right);

    double last = intervals[0].right;
    int count = 1;
    for (int i = 1; i < n; i++) {
        if (intervals[i].left > last) {
            last = intervals[i].right;
            count++;
        }
    }
    return count;
}

In [5]:
int main(void)
{
    int k;
    if (scanf("%d", &k) != 1) return 1;

    for (int group = 0; group < k; group++) {
        int n;
        double radius;
        BrightRegion regions[100];

        if (scanf("%d %lf", &n, &radius) != 2) return 1;
        if (n < 1 || n > 100) return 1;

        for (int i = 0; i < n; i++) {
            if (scanf("%lf %lf", &regions[i].x, &regions[i].y) != 2) return 1;
        }

        printf("%d\n", minimum_stations(regions, n, radius));
    }
    return 0;
}

## 测试用例

先检查亮区到区间的换算和右端点比较，再用独立数据调用 `minimum_stations`。每组将实际结果与手算值对照。

In [6]:
{
    const BrightRegion point = {6, -4};
    const Interval interval = coverage_interval(point, 5);
    printf("亮区 (6, -4)：实际 [%g, %g]，预期 [3, 9]，%s\n",
           interval.left, interval.right,
           interval.left == 3 && interval.right == 9 ? "通过" : "失败");
}

亮区 (6, -4)：实际 [3, 9]，预期 [3, 9]，通过


In [7]:
{
    const Interval a = coverage_interval((BrightRegion){2, 2}, 2);
    const Interval b = coverage_interval((BrightRegion){0, 1}, 2);
    // 2 与 √3 相差不足 1，但两个右端点的排序顺序仍然确定。
    const int follows = compare_by_right(&a, &b) > 0;
    printf("小数端点比较：实际 %.3f 和 %.3f，前者排在后面：%s，预期是，%s\n",
           a.right, b.right, follows ? "是" : "否", follows ? "通过" : "失败");
}

小数端点比较：实际 2.000 和 1.732，前者排在后面：是，预期是，通过


In [8]:
static void check_supply_case(const char *name, const BrightRegion regions[],
                              int n, double radius, int expected)
{
    int actual = minimum_stations(regions, n, radius);
    printf("%s：实际 %d，预期 %d，%s\n",
           name, actual, expected, actual == expected ? "通过" : "失败");
}

In [9]:
{
    // 两个区间共用横坐标 3。
    const BrightRegion regions[] = {
        {0, 4},
        {6, -4}
    };
    check_supply_case("共用闭区间端点", regions, 2, 5, 1);
}

共用闭区间端点：实际 1，预期 1，通过


In [10]:
{
    // 三个亮区只能分别由 -8、0、8 处的站点覆盖。
    const BrightRegion regions[] = {
        {-8, 5},
        {0, -5},
        {8, 5}
    };
    check_supply_case("三个不同的单点区间", regions, 3, 5, 3);
}

三个不同的单点区间：实际 3，预期 3，通过


In [11]:
{
    // 一个站点就能覆盖唯一的亮区。
    const BrightRegion regions[] = {
        {2, 0}
    };
    check_supply_case("单个亮区位于 X 轴", regions, 1, 5, 1);
}

单个亮区位于 X 轴：实际 1，预期 1，通过


In [12]:
{
    // 最右侧区间最先输入；其余两个区间可以共用站点 1。
    const BrightRegion regions[] = {
        {10, 0},
        {0, 0},
        {2, 0}
    };
    check_supply_case("乱序输入", regions, 3, 1, 2);
}

乱序输入：实际 2，预期 2，通过


In [13]:
{
    // 单点 2 位于区间 [-√3, √3] 之外。
    const BrightRegion regions[] = {
        {2, 2},
        {0, 1}
    };
    check_supply_case("小数端点之间有间隙", regions, 2, 2, 2);
}

小数端点之间有间隙：实际 2，预期 2，通过


In [14]:
{
    BrightRegion regions[100];
    for (int i = 0; i < 100; i++) {
        // 横坐标互不相同，且每个亮区到 X 轴的距离恰好等于半径。
        regions[i] = (BrightRegion){49 - i, 1};
    }
    check_supply_case("100 个单点区间", regions, 100, 1, 100);
}

100 个单点区间：实际 100，预期 100，通过


### 多组标准输入输出

将前五组用例一次输入编译后的 [C23 源程序](supply_system.c)，同时检验组数读取、逐组求解和换行输出。

**标准输入**
```text
5
2 5
0 4
6 -4
3 5
-8 5
0 -5
8 5
1 5
2 0
3 1
10 0
0 0
2 0
2 2
2 2
0 1
```

**实际标准输出**
```text
1
3
1
2
2
```

## 复杂度分析与实验总结

### 单组时间复杂度

设本组有 $n$ 个亮区，按固定精度算术和一次开方均耗费常数时间分析。

| 阶段 | 操作次数及耗时 |
| --- | --- |
| 生成区间 | 对每个亮区调用一次 `coverage_interval`，共 $n$ 次，耗时 $\Theta(n)$。 |
| 按右端点排序 | 每次比较两个右端点耗时 $O(1)$。记排序耗时为 $T_{\mathrm{sort}}(n)$。 |
| 贪心扫描 | 取首个右端点建站后，恰好比较 $n-1$ 次。能否沿用站点只影响更新次数，扫描连同初始化耗时 $\Theta(n)$。 |

因此 $T(n)=\Theta(n)+T_{\mathrm{sort}}(n)$。在排序耗时为 $O(n\log n)$ 的假设下，单组总时间复杂度为 $O(n\log n)$。读入本组的 $n$ 对坐标另需 $\Theta(n)$ 时间。

### 多组时间复杂度

设第 $i$ 组有 $n_i$ 个亮区。程序逐组读入并独立求解，各组耗时相加。在上述排序假设下，总时间复杂度为

$$
O\!\left(\sum_{i=1}^{K}\bigl(n_i+n_i\log n_i\bigr)\right).
$$

保留线性项也涵盖了 $n_i=1$ 的情况。即使本组只有一个亮区，程序仍需读取坐标、生成区间并设置首站。

### 实验总结

选址的关键是把每个亮区对应到 X 轴上的一个闭区间。按右端点从小到大扫描时，只有当前区间的左端点严格大于最近的站点横坐标，才需要新建站。

第一组中的 $[-3,3]$ 和 $[3,9]$ 可以共用端点 $3$。用同一份输入编译运行两个版本，正式程序的 `left > last` 输出 $1$。在临时对照版本中改用 `left >= last`，第一组输出变为 $2$。此时第二个区间的左端点已经有站，多出的一站来自对相等情况的误判。